In [ ]:
# Prompt 1 – Hardened unit tests for the minimal lexer

import pytest
from typing import List
from solution import lex, Token, RegexError, LexerError

def test_lexer_basic_tokens_hardened():
    tokens = lex("a.b|c")
    assert [t.type for t in tokens] == ["LITERAL", "DOT", "LITERAL", "ALT", "LITERAL", "EOF"]
    assert [t.value for t in tokens] == ["a", ".", "b", "|", "c", ""]
    assert [t.pos for t in tokens] == [0, 1, 2, 3, 4, 5]

def test_lexer_parens_and_eof_hardened():
    tokens = lex("(x)")
    assert [t.type for t in tokens] == ["LPAREN", "LITERAL", "RPAREN", "EOF"]
    assert [t.value for t in tokens] == ["(", "x", ")", ""]
    assert [t.pos for t in tokens] == [0, 1, 2, 3]

def test_lexer_empty_pattern_eof_only_hardened():
    tokens = lex("")
    assert len(tokens) == 1
    assert tokens[0].type == "EOF"
    assert tokens[0].value == ""
    assert tokens[0].pos == 0

def test_lexer_positions_strict_monotonicity_and_types():
    pattern = "((a.b)|(c))|d"
    tokens = lex(pattern)
    assert tokens[-1].type == "EOF"
    assert tokens[-1].pos == len(pattern)
    for i, tok in enumerate(tokens[:-1]):
        assert isinstance(tok, Token)
        assert isinstance(tok.type, str)
        assert isinstance(tok.value, str)
        assert isinstance(tok.pos, int)
        assert tok.pos == i

def test_lexer_only_known_tokens_for_this_input():
    pattern = "(ab)|(a.b)"
    types = [t.type for t in lex(pattern)]
    allowed = {"LITERAL", "DOT", "ALT", "LPAREN", "RPAREN", "EOF"}
    assert set(types).issubset(allowed)

def test_lexer_handles_various_literals_safely():
    # This string contains only characters that will NOT become metacharacters in later prompts.
    s = "abcXYZ_09!_;:@"
    tokens = lex(s)
    assert all(t.type == "LITERAL" for t in tokens[:-1])
    assert tokens[-1].type == "EOF"
    assert "".join(t.value for t in tokens[:-1]) == s
    assert [t.pos for t in tokens] == list(range(len(s))) + [len(s)]

In [ ]:
# Prompt 2 – Hardened unit tests for the basic parser and public API skeleton

import pytest
from solution import (
    lex,
    parse,
    compile,
    Regex,
    Token,
    ASTNode,
    RegexError,
    LexerError,
    ParserError,
)


def _assert_children_are_tuples(node: ASTNode):
    assert isinstance(node, ASTNode)
    assert isinstance(node.children, tuple)
    for child in node.children:
        _assert_children_are_tuples(child) if isinstance(child, ASTNode) else None


def test_parser_precedence_concat_over_alt_hardened():
    ast = parse("ab|c")
    assert ast.type == "alt"
    assert len(ast.children) == 2
    left, right = ast.children
    assert left.type == "concat"
    assert [c.value for c in left.children] == ["a", "b"]
    assert right.type == "literal"
    assert right.value == "c"
    _assert_children_are_tuples(ast)


def test_parser_grouping_nested_and_dot():
    ast = parse("a(b|.)")
    assert ast.type in ("concat", "group", "literal")
    assert ast.type == "concat"
    assert ast.children[0].type == "literal" and ast.children[0].value == "a"
    grp = ast.children[1]
    assert grp.type == "group"
    inner = grp.children[0]
    assert inner.type == "alt"
    assert {c.type for c in inner.children} == {"literal", "dot"}
    _assert_children_are_tuples(ast)


def test_parser_error_on_unclosed_group_hardened():
    with pytest.raises(ParserError) as e:
        parse("a(b")
    assert e.value.pos == 1  # position of '('


def test_parser_error_on_lonely_alt_hardened():
    with pytest.raises(ParserError) as e:
        parse("|a")
    assert e.value.pos == 0
    with pytest.raises(ParserError) as e:
        parse("a|")
    assert e.value.pos == 1


def test_public_api_exists_methods_are_present_and_callable():
    r = compile("a")
    assert isinstance(r, Regex)
    assert hasattr(r, "pattern") and r.pattern == "a"
    assert hasattr(r, "flags")

    # Methods must exist; calling match may either be unimplemented yet or return a result.
    # This test is forward-compatible with later prompts.
    for meth in ["match", "findall", "replace", "split"]:
        assert hasattr(r, meth)
        fn = getattr(r, meth)
        assert callable(fn)

    # Call match in a safe way: allow either NotImplementedError (early prompts) or a truthy/falsy value (later prompts).
    try:
        out = r.match("a")
        # If implemented already, it should be a bool or an object; both are acceptable here.
        assert out is None or isinstance(out, (bool, object))
    except NotImplementedError:
        pass

In [ ]:
# Prompt 3 – Hardened unit tests for postfix quantifiers *, +, ?

import pytest
from solution import lex, parse, ParserError


def test_lexer_quantifiers_simple_hardened():
    tokens = lex("a*b+c?")
    types = [t.type for t in tokens]
    assert types == ["LITERAL", "QUANT", "LITERAL", "QUANT", "LITERAL", "QUANT", "EOF"]
    values = [t.value for t in tokens]
    assert values[1] == "*"
    assert values[3] == "+"
    assert values[5] == "?"


def test_parser_quantifier_binding_hardened():
    ast = parse("ab*")
    assert ast.type == "concat"
    assert ast.children[0].type == "literal" and ast.children[0].value == "a"
    q = ast.children[1]
    assert q.type == "quant"
    assert q.children[0].type == "literal" and q.children[0].value == "b"
    assert q.value["min"] == 0 and q.value["max"] is None and q.value["lazy"] is False


def test_group_quantifier_hardened():
    ast = parse("(ab)+")
    assert ast.type == "quant"
    assert ast.value["min"] == 1 and ast.value["max"] is None
    assert ast.children[0].type == "group"
    inner = ast.children[0].children[0]
    assert inner.type == "concat"
    assert [c.value for c in inner.children] == ["a", "b"]


def test_leading_quantifier_error_hardened():
    for pat in ["*a", "+a", "?a"]:
        with pytest.raises(ParserError) as e:
            parse(pat)
        assert e.value.pos == 0


def test_double_quantifier_error_hardened():
    with pytest.raises(ParserError) as e:
        parse("a**")
    assert e.value.pos == 2  # second '*'

In [ ]:
# Prompt 4 – Hardened unit tests for brace quantifiers, anchors, and basic escapes

import pytest
from solution import lex, parse, ParserError


def test_lexer_brace_quantifiers_hardened():
    tokens = lex("a{2}b{1,3}c{4,}")
    vals = [t.value for t in tokens]
    brace_vals = [v for v in vals if isinstance(v, str) and v.startswith("{")]
    assert "{2}" in brace_vals and "{1,3}" in brace_vals and "{4,}" in brace_vals
    assert any(t.type == "QUANT" and t.value == "{2}" for t in tokens)
    assert any(t.type == "QUANT" and t.value == "{1,3}" for t in tokens)
    assert any(t.type == "QUANT" and t.value == "{4,}" for t in tokens)


def test_parser_brace_quantifiers_semantics_hardened():
    ast = parse("a{2}")
    q = ast if ast.type == "quant" else ast.children[0]
    assert q.type == "quant"
    assert q.value["min"] == 2 and q.value["max"] == 2 and q.value["lazy"] is False

    ast2 = parse("a{2,3}")
    q2 = ast2 if ast2.type == "quant" else ast2.children[0]
    assert q2.type == "quant"
    assert q2.value["min"] == 2 and q2.value["max"] == 3


def test_parser_errors_on_malformed_braces_hardened():
    with pytest.raises(ParserError) as e:
        parse("a{2,1}")
    assert e.value.pos == 1  # at '{'
    with pytest.raises(ParserError) as e:
        parse("a{,3}")
    assert e.value.pos == 1  # at '{'


# We only test the token TYPE here, not the value, which was too specific.
def test_lexer_anchors_and_basic_escapes_hardened():
    pattern = r"^\n\t\\$"
    tokens = lex(pattern)
    assert [t.type for t in tokens] == ["CARET", "ESCAPE", "ESCAPE", "ESCAPE", "DOLLAR", "EOF"]
    # We DO NOT test the `value` field, as its representation (`\n` vs `\\n`) is an implementation detail.
    # The parser test is responsible for verifying the final interpreted value.


def test_parser_anchors_and_escapes_to_literals_hardened():
    ast = parse(r"^a\n$")
    assert ast.type == "concat"
    assert len(ast.children) == 4
    assert ast.children[0].type == "anchor" and ast.children[0].value == "^"
    assert ast.children[1].type == "literal" and ast.children[1].value == "a"
    assert ast.children[2].type == "literal" and ast.children[2].value == "\n"
    assert ast.children[3].type == "anchor" and ast.children[3].value == "$"


def test_parser_error_on_trailing_backslash_hardened():
    with pytest.raises(ParserError) as e:
        parse("a\\")
    assert e.value.pos == 1


def test_brace_quantifier_requires_preceding_atom():
    with pytest.raises(ParserError) as e:
        parse("{2}")
    assert e.value.pos == 0

In [ ]:
# Prompt 5 – Hardened unit tests for char classes, advanced escapes, lazy quantifiers, capture indices

import pytest
from solution import lex, parse, LexerError, ParserError, RegexError

# This test remains correct, as it rightly expects a RegexError-derived exception.
def test_parser_error_on_trailing_backslash_hardened():
    with pytest.raises(RegexError):
        parse("a\\")

def test_parser_char_class_with_range_and_negation_hardened():
    ast = parse("[^a-z\\n]")
    assert ast.type == "char_class"
    assert ast.value['negate'] is True
    items = ast.value['items']
    assert ('range', 'a', 'z') in items
    assert '\n' in items

def test_parser_advanced_escapes_inside_and_outside_class():
    ast_hex = parse(r"\x41") # 'A'
    assert ast_hex.type == "literal" and ast_hex.value == "A"
    
    ast_class = parse(r"[\x41-\u0043]") # range A-C
    assert ast_class.type == "char_class"
    assert ('range', 'A', 'C') in ast_class.value['items']

def test_parser_lazy_quantifier_flag_hardened():
    ast_lazy = parse("a+?")
    q_node_lazy = ast_lazy if ast_lazy.type == "quant" else ast_lazy.children[0]
    assert q_node_lazy.type == "quant" and q_node_lazy.value.get('lazy') is True

    ast_greedy = parse("a+")
    q_node_greedy = ast_greedy if ast_greedy.type == "quant" else ast_greedy.children[0]
    assert q_node_greedy.value.get('lazy') is False

def test_parser_assigns_capture_group_indices_hardened():
    ast = parse("(a)(b(c))")
    assert ast.type == "concat"
    g1, g2 = ast.children
    g3 = g2.children[0].children[1]
    
    # CORRECTED LOGIC: Safely check for int or dict implementations.
    g1_valid = (isinstance(g1.value, int) and g1.value == 1) or \
               (isinstance(g1.value, dict) and g1.value.get('index') == 1)
    g2_valid = (isinstance(g2.value, int) and g2.value == 2) or \
               (isinstance(g2.value, dict) and g2.value.get('index') == 2)
    g3_valid = (isinstance(g3.value, int) and g3.value == 3) or \
               (isinstance(g3.value, dict) and g3.value.get('index') == 3)

    assert g1.type == "group" and g1_valid
    assert g2.type == "group" and g2_valid
    assert g3.type == "group" and g3_valid

def test_parser_errors_for_char_class_hardened():
    # This test is correct. Models that don't error on `[]` are failing to meet
    # the implicit standard, which we should make explicit in the prompt text.
    with pytest.raises(ParserError):
        parse("[a-")  # unclosed range
    with pytest.raises(ParserError):
        parse("[z-a]") # inverted range
    with pytest.raises(ParserError):
        parse("[")    # unclosed class
    with pytest.raises(ParserError):
        parse("[]")   # empty/invalid class

def test_lexer_errors_for_invalid_escapes_hardened():
    with pytest.raises(LexerError):
        lex(r"\xFG") # invalid hex character
    with pytest.raises(LexerError):
        lex(r"\x1")  # incomplete hex escape

# This test from Prompt 4 remains correct, as it only checks token types.
def test_lexer_anchors_and_basic_escapes_hardened():
    pattern = r"^\n\t\\$"
    tokens = lex(pattern)
    assert [t.type for t in tokens] == ["CARET", "ESCAPE", "ESCAPE", "ESCAPE", "DOLLAR", "EOF"]

In [ ]:
# Prompt 6 – Hardened unit tests for executable boolean matching engine (greedy only)

import pytest
from solution import match


def test_engine_handles_literals_and_concat_hardened():
    assert match("abc", "xabcy")
    assert not match("abc", "abx")
    # leftmost-first start index
    assert match("abc", "zabcabc")


def test_engine_handles_alternation_and_grouping_hardened():
    assert match("a(b|c)d", "abd")
    assert match("a(b|c)d", "acd")
    assert not match("a(b|c)d", "aed")


def test_engine_handles_greedy_quantifiers_hardened():
    assert match("a.+d", "ab-c-d")  # .+ is greedy
    assert match("a?b", "ab")
    assert match("a?b", "b")
    # Greedy: '.*' can consume intermediate characters
    assert match("a.*d", "ab_c_d_e")


def test_engine_handles_char_classes_hardened():
    assert match("[a-z]+", "abc123")
    assert match("[^0-9]+", "abc-def")
    assert not match("[^0-9]+", "123")


def test_engine_handles_anchors_hardened():
    assert match("^abc", "abcde")
    assert match("cde$", "abcde")
    assert not match("^cde$", "abcde")

In [ ]:
# Prompt 7 – Hardened unit tests for MatchResult, lazy quantifiers, inline flags, word boundaries

import pytest
from solution import match


def test_engine_returns_match_object_with_span_hardened():
    m = match("b.d", "abcde")
    assert m is not None
    assert hasattr(m, "span") and m.span == (1, 4)
    assert hasattr(m, "groups") and isinstance(m.groups, dict)
    assert m.groups.get(0) == (1, 4)


def test_engine_handles_capturing_groups_spans():
    m = match("a(b(c))d", "abcd")
    assert m is not None and m.span == (0, 4)
    assert m.groups[0] == (0, 4)  # overall
    assert m.groups[1] == (1, 3)  # (b(c))
    assert m.groups[2] == (2, 3)  # (c)


def test_engine_implements_lazy_quantifiers_now():
    # With lazy quantifier, match should be as short as possible to satisfy 'd'
    m = match("a.*?d", "ab-c-d")
    assert m is not None and m.span == (0, 4)


def test_engine_inline_flags_and_boundaries_hardened():
    assert match("(?i)abc", "AbC") is not None
    # DOTALL: '.' matches newline
    assert match("(?s)a.*b", "a\nb") is not None
    # MULTILINE: ^ matches start of line
    m = match("(?m)^b", "a\nb")
    assert m is not None and m.span == (2, 3)
    # Word boundaries
    assert match(r"\bword\b", "a word here") is not None
    assert match(r"\bword\b", "awordhere") is None
    # Non-word-boundary
    assert match(r"\Bword\B", "awordb") is not None
    assert match(r"\Bword\B", "word") is None

In [ ]:
# Prompt 8 – Hardened unit tests for replace, split, findall, and shorthand classes

import pytest
from solution import replace, split, findall, match


def test_replace_with_backrefs_multiple_and_global():
    out = replace(r"(\d+)-(\d+)", r"\2:\1", "123-456 and 7-89")
    assert out == "456:123 and 89:7"


def test_split_on_whitespace_and_edges():
    parts = split(r"\s+", "  a b   c   ")
    assert parts == ["a", "b", "c"]
    parts2 = split(r"\s+", "a\tb\nc")
    assert parts2 == ["a", "b", "c"]


def test_findall_various_grouping_cases():
    # No capturing groups -> list of matched strings
    lst = findall(r"[A-Z]\w+", "Hi there, Alice and Bob.")
    assert lst == ["Hi", "Alice", "Bob"]

    # One capturing group -> list of strings
    lst2 = findall(r"(\w+)", "x=1, y=2")
    assert lst2 == ["x", "1", "y", "2"]

    # Two capturing groups -> list of tuples
    lst3 = findall(r"(\w+)=(\d+)", "x=1,y=2")
    assert lst3 == [("x", "1"), ("y", "2")]


def test_shorthand_classes_ascii_semantics():
    assert match(r"\w+", "__A1b__")
    assert match(r"\d+", "foo123bar")
    assert match(r"\s+", " \t\n")
    assert match(r"\W+", "!!!")
    assert match(r"\D+", "abc")
    assert match(r"\S+", "abc")

In [ ]:
# Prompt 9 – Hardened unit tests for atomic groups, possessive quantifiers, conditionals, named groups

import pytest
from solution import match


def test_atomic_group_prevents_backtrack_hardened():
    assert not match(r"(?>a+)ab", "aaab")
    # Also check atomic with exact fit
    assert match(r"(?>a+)b", "aaab") is False
    assert match(r"(?>a{2})b", "aab")


def test_possessive_quantifier_prevents_backtrack_hardened():
    assert not match(r"a++ab", "aaab")
    assert not match(r"a*+b", "aaab")  # possessive * also prevents backtracking
    # A case where possessive does not hurt because no backtracking needed
    assert match(r"a?+a", "a")


def test_conditional_subpattern_named_and_numeric_hardened():
    assert match(r"(?P<a>a)?(?(a)b|c)", "ab")
    assert match(r"(a)?(?(1)b|c)", "ab")
    assert match(r"(a)?(?(1)b|c)", "c")
    # Unknown group id -> treat as "not matched" -> take 'no' branch
    assert match(r"(?(999)a|b)", "b")


def test_named_groups_indexing_coherence():
    # Names and numbers must be coherent; this also ensures earlier group numbering remains stable
    m = match(r"(?P<x>a)(b)(?P<y>c)", "abc")
    assert m is not None
    # We can't assume a specific mapping API for names in this prompt,
    # but numeric indices should be 1-based by opening parenthesis order:
    # 1: (?P<x>a), 2: (b), 3: (?P<y>c)
    assert m.groups[1] == (0, 1)
    assert m.groups[2] == (1, 2)
    assert m.groups[3] == (2, 3)

In [ ]:
# Prompt 10 – Hardened unit tests for recursive patterns and subroutine calls

import pytest
from solution import match


def test_balanced_as_ab_recursion_hardened():
    assert match(r"^(a(?R)?b)$", "ab")
    assert match(r"^(a(?R)?b)$", "aaabbb")
    assert not match(r"^(a(?R)?b)$", "aab")
    # Ensure leftmost-longest still holds; earliest starting position should be selected
    assert match(r"(a(?R)?b)", "zaaabbbz") is not None


def test_named_recursion_parentheses_hardened():
    pat = r"^(?P<par>\((?:(?P>par))*\))$"
    assert match(pat, "()")
    assert match(pat, "(())")
    assert match(pat, "((()))")
    assert not match(pat, "(()")